# Dynamic Prompts

`@dynamic_prompt` middleware builds the system prompt from runtime context (here: preferred reply language).

**Key learnings**
- `context_schema` defines typed runtime context passed into `invoke`
- Middleware reads `request.runtime.context` and returns a prompt string
- Same agent can serve many locales without rebuilding


In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Context + dynamic prompt

`LanguageContext.user_language` drives whether the assistant must reply in another language.


In [ ]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on the user's preferred language."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    return base_prompt


### Create the agent

Register the context schema and middleware; no static system prompt needed.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

### Try Irish, Spanish, and French

Pass a different `LanguageContext` on each invoke and compare the replies.


In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)